# 기술 스택 추출 파이프라인

세 개의 스크립트를 작업 순서대로 합친 노트북입니다.

1. **tech_dictionary.py** — canonical 기술명과 별칭(alias) 사전 정의
2. **mine_tech_candidates.py** — 사전에 없는 기술스택 후보를 채용공고 텍스트에서 마이닝
3. **extract_tech_tags.py** — 사전을 기반으로 description 열에서 실제 태그 추출


In [1]:
import re
from collections import Counter

import pandas as pd


## 1. 기술 스택 사전 (`tech_dictionary.py`)

구조: `category -> { canonical_name: [alias1, alias2, ...] }`
- `canonical_name`은 매칭 후 tags 열에 실제로 들어갈 표준 이름.
- `aliases`에는 canonical_name 자체를 포함해서 적어야 함(자동으로 추가되지 않음).
- 대소문자/표기 변형(JS, Javascript 등)은 alias로 추가.

한 글자~두 글자짜리 애매한 언어명(Go, R, C, D)은 `AMBIGUOUS_EXACT`에 등록해서
다음 단계(`extract_tech_tags`)에서 대소문자 구분 + 엄격한 경계로만 매칭한다.
이 사전은 시작점일 뿐이며, 2단계에서 뽑은 후보를 검토해서 계속 늘려가는 것을 전제로 한다.


In [2]:
TECH_DICTIONARY = {
    "language": {
        "Python": ["Python"],
        "Java": ["Java"],
        "JavaScript": ["JavaScript", "Javascript", "JS"],
        "TypeScript": ["TypeScript", "Typescript", "TS"],
        "Go": ["Golang", "Go"],
        "Rust": ["Rust"],
        "C++": ["C++"],
        "C#": ["C#"],
        "C": ["C"],
        "Ruby": ["Ruby"],
        "PHP": ["PHP"],
        "Swift": ["Swift"],
        "Kotlin": ["Kotlin"],
        "Scala": ["Scala"],
        "R": ["R"],
        "Objective-C": ["Objective-C", "Objective C", "ObjC"],
        "Dart": ["Dart"],
        "Elixir": ["Elixir"],
        "Erlang": ["Erlang"],
        "Haskell": ["Haskell"],
        "Perl": ["Perl"],
        "MATLAB": ["MATLAB", "Matlab"],
        "SQL": ["SQL"],
        "Bash": ["Bash", "Bash scripting", "Shell scripting"],
        "Julia": ["Julia"],
        "Groovy": ["Groovy"],
        "Lua": ["Lua"],
        "Solidity": ["Solidity"],
        "D": ["D"],
    },
    "frontend": {
        "React": ["React", "React.js", "ReactJS"],
        "Angular": ["Angular", "AngularJS", "Angular.js"],
        "Vue.js": ["Vue.js", "Vue", "VueJS"],
        "Svelte": ["Svelte", "SvelteKit"],
        "Next.js": ["Next.js", "NextJS"],
        "Nuxt.js": ["Nuxt.js", "NuxtJS"],
        "jQuery": ["jQuery"],
        "Redux": ["Redux"],
        "HTML": ["HTML", "HTML5"],
        "CSS": ["CSS", "CSS3"],
        "Sass": ["Sass", "SCSS"],
        "Tailwind CSS": ["Tailwind CSS", "TailwindCSS", "Tailwind"],
        "Webpack": ["Webpack"],
        "Vite": ["Vite"],
        "Ember.js": ["Ember.js", "EmberJS"],
        "Backbone.js": ["Backbone.js", "BackboneJS"],
        "D3.js": ["D3.js", "D3JS"],
        "Storybook": ["Storybook"],
    },
    "backend": {
        "Node.js": ["Node.js", "NodeJS", "Node"],
        "Express": ["Express.js", "ExpressJS", "Express"],
        "Django": ["Django"],
        "Flask": ["Flask"],
        "FastAPI": ["FastAPI"],
        "Spring": ["Spring Boot", "Spring Framework", "SpringBoot"],
        "Ruby on Rails": ["Ruby on Rails", "Rails"],
        "Laravel": ["Laravel"],
        ".NET": [".NET", "ASP.NET", "dotnet"],
        "NestJS": ["NestJS", "Nest.js"],
        "gRPC": ["gRPC"],
        "GraphQL": ["GraphQL"],
        "REST": ["REST API", "RESTful"],
    },
    "database": {
        "PostgreSQL": ["PostgreSQL", "Postgres"],
        "MySQL": ["MySQL"],
        "MongoDB": ["MongoDB", "Mongo"],
        "Redis": ["Redis"],
        "Cassandra": ["Cassandra"],
        "DynamoDB": ["DynamoDB"],
        "Elasticsearch": ["Elasticsearch", "ElasticSearch"],
        "SQLite": ["SQLite"],
        "Oracle DB": ["Oracle Database", "Oracle DB"],
        "MariaDB": ["MariaDB"],
        "Snowflake": ["Snowflake"],
        "BigQuery": ["BigQuery"],
        "Redshift": ["Redshift"],
        "ClickHouse": ["ClickHouse"],
        "Neo4j": ["Neo4j"],
        "CockroachDB": ["CockroachDB"],
        "Firestore": ["Firestore"],
        "MongoDB Atlas": ["MongoDB Atlas"],
    },
    "cloud_devops": {
        "AWS": ["AWS", "Amazon Web Services"],
        "GCP": ["GCP", "Google Cloud Platform", "Google Cloud"],
        "Azure": ["Azure", "Microsoft Azure"],
        "Docker": ["Docker"],
        "Kubernetes": ["Kubernetes", "K8s"],
        "Terraform": ["Terraform"],
        "Ansible": ["Ansible"],
        "Jenkins": ["Jenkins"],
        "GitHub Actions": ["GitHub Actions"],
        "GitLab CI": ["GitLab CI", "GitLab CI/CD"],
        "CircleCI": ["CircleCI"],
        "Helm": ["Helm"],
        "Istio": ["Istio"],
        "Prometheus": ["Prometheus"],
        "Grafana": ["Grafana"],
        "Datadog": ["Datadog"],
        "Nginx": ["Nginx"],
        "Linux": ["Linux"],
        "Cloudflare": ["Cloudflare"],
        "Vault": ["HashiCorp Vault", "Vault"],
        "OpenTelemetry": ["OpenTelemetry"],
        "AWS CloudFormation": ["CloudFormation"],
        "Amazon EKS": ["EKS", "Amazon EKS"],
        "Google GKE": ["GKE", "Google Kubernetes Engine"],
        "Amazon EC2": ["EC2", "Amazon EC2"],
        "Amazon S3": ["S3", "Amazon S3"],
        "ArgoCD": ["ArgoCD", "Argo CD"],
        "Buildkite": ["Buildkite"],
        "OpenShift": ["OpenShift"],
        "Pulumi": ["Pulumi"],
        "Dependabot": ["Dependabot"],
        "Puppet": ["Puppet"],
        "Snyk": ["Snyk"],
        "Okta": ["Okta"],
        "Splunk": ["Splunk"],
        "WebAssembly": ["WebAssembly", "Wasm"],
    },
    "data_ml": {
        "TensorFlow": ["TensorFlow"],
        "PyTorch": ["PyTorch"],
        "Keras": ["Keras"],
        "scikit-learn": ["scikit-learn", "sklearn"],
        "Pandas": ["Pandas"],
        "NumPy": ["NumPy"],
        "Apache Spark": ["Apache Spark", "Spark", "PySpark"],
        "Hadoop": ["Hadoop"],
        "Apache Kafka": ["Apache Kafka", "Kafka"],
        "Airflow": ["Airflow", "Apache Airflow"],
        "dbt": ["dbt"],
        "Databricks": ["Databricks"],
        "Jupyter": ["Jupyter", "Jupyter Notebook"],
        "Hugging Face": ["Hugging Face", "HuggingFace"],
        "Transformers": ["Transformers"],
        "LangChain": ["LangChain"],
        "OpenCV": ["OpenCV"],
        "MLflow": ["MLflow"],
        "XGBoost": ["XGBoost"],
        "LightGBM": ["LightGBM"],
        "Tableau": ["Tableau"],
        "Power BI": ["Power BI", "PowerBI"],
        "Looker": ["Looker"],
        "Apache Flink": ["Apache Flink", "Flink"],
        "Apache Iceberg": ["Apache Iceberg", "Iceberg"],
        "Trino": ["Trino"],
        "Dagster": ["Dagster"],
        "Vertex AI": ["Vertex AI", "Vertex"],
        "Triton Inference Server": ["Triton"],
        "JAX": ["JAX"],
        "CUDA": ["CUDA"],
        "NCCL": ["NCCL"],
        "GitHub Copilot": ["Copilot", "GitHub Copilot"],
        "OpenAI Codex": ["Codex"],
        "MCP": ["MCP", "Model Context Protocol"],
        "Slurm": ["Slurm"],
        "Bazel": ["Bazel"],
        "Gradle": ["Gradle"],
    },
    "mobile": {
        "iOS": ["iOS"],
        "Android": ["Android"],
        "React Native": ["React Native"],
        "Flutter": ["Flutter"],
        "SwiftUI": ["SwiftUI"],
        "Xamarin": ["Xamarin"],
        "Jetpack Compose": ["Jetpack Compose"],
    },
    "messaging_infra": {
        "RabbitMQ": ["RabbitMQ"],
        "Apache Pulsar": ["Apache Pulsar", "Pulsar"],
        "gRPC": ["gRPC"],
        "Zookeeper": ["Zookeeper"],
    },
    "tools": {
        "Git": ["Git"],
        "GitHub": ["GitHub"],
        "GitLab": ["GitLab"],
        "Jira": ["Jira"],
        "Figma": ["Figma"],
        "Postman": ["Postman"],
        "Confluence": ["Confluence"],
        "Salesforce": ["Salesforce", "Salesforce.com", "SFDC"],
        "HubSpot": ["HubSpot", "Hubspot"],
        "Zendesk": ["Zendesk"],
        "NetSuite": ["NetSuite", "Netsuite"],
        "Vanta": ["Vanta"],
    },
    "ai_platform": {
        "OpenAI": ["OpenAI"],
        "Anthropic": ["Anthropic"],
        "Claude": ["Claude"],
        "GPT": ["GPT", "GPT-4", "ChatGPT"],
        "Gemini": ["Gemini"],
        "Llama": ["Llama"],
    },
    "testing": {
        "Jest": ["Jest"],
        "PyTest": ["PyTest", "pytest"],
        "Selenium": ["Selenium"],
        "Cypress": ["Cypress"],
        "JUnit": ["JUnit"],
    },
}

# 대소문자를 반드시 구분해서, "엄격한" 경계로만 매칭할 애매한 짧은 토큰.
# 예: "Go" vs 일반 동사 "go", "R" vs "R&D"의 R, "C" vs 성적 등급 C.
# 완벽하지 않으니 추출 후 결과를 한 번 훑어보고 오탐이 많으면 여기서 빼거나
# extract_tech_tags 의 매칭 규칙을 더 강화할 것.
AMBIGUOUS_EXACT = {"Go", "R", "C", "D"}


## 2. 기술스택 후보 마이닝 (`mine_tech_candidates.py`)

`TECH_DICTIONARY`에 아직 없는 "기술스택처럼 생긴" 토큰의 빈도를 뽑아서
사전을 늘려나갈 때 검토용 후보 리스트를 만든다.

이 단계는 자동으로 사전에 추가하지 않는다. 출력된 후보를 사람이 훑어보고
진짜 기술스택만 골라 위 1단계 `TECH_DICTIONARY`에 직접 추가하는 것을 전제로 한다.


In [3]:
# PostgreSQL, GraphQL, TensorFlow 같은 CamelCase / Node.js 같은 dotted 이름을 후보로 포착.
# 대문자로 시작 + 영숫자, 뒤에 ".js" 류 확장이나 "++","#" 접미사가 붙는 경우까지 허용.
CANDIDATE_PATTERN = re.compile(
    r"\b[A-Z][A-Za-z0-9]{1,}(?:\.[A-Za-z]{1,5})?(?:\+\+|#)?\b"
)
# candidate와 동일한 단어가 완전 소문자로도 등장하는 빈도를 재는 용도.
# "Build"/"Design"/"Learn" 처럼 불릿·문장 맨 앞이라 우연히 대문자가 된 일반 단어는
# 본문 다른 곳에서 소문자로도 흔히 쓰이지만, React/Kubernetes 같은 진짜 고유명사는 거의 항상 대문자로만 등장한다.
LOWERCASE_WORD_PATTERN = re.compile(r"\b[a-z]+\b")

# 채용공고에 아주 흔히 등장해서 후보로 나와봤자 노이즈인 단어들.
# 완전한 목록이 될 수 없으니 실제 출력을 보고 계속 추가해서 쓸 것.
STOPWORDS = {
    "The", "We", "Our", "You", "Your", "This", "That", "These", "Those", "As", "In", "At",
    "With", "For", "And", "Or", "But", "Is", "Are", "Was", "Were", "Be", "Been", "Being",
    "It", "Its", "If", "Then", "So", "Team", "Role", "About", "Who", "What", "When", "Where",
    "Why", "How", "Job", "Company", "Work", "Join", "Us", "Will", "Can", "May", "Must",
    "Should", "Would", "Could", "Please", "Note", "New", "All", "Most", "Some", "Any", "Each",
    "Every", "Other", "Also", "More", "Than", "Such", "Not", "No", "Yes", "Ok", "Well", "Good",
    "Great", "Strong", "Excellent", "Location", "Salary", "Benefits", "Requirements",
    "Qualifications", "Responsibilities", "Experience", "Skills", "Knowledge", "Ability",
    "Years", "Degree", "Bachelor", "Master", "Phd", "Apply", "Application", "Candidate",
    "Candidates", "Employer", "Employment", "Equal", "Opportunity", "Diversity", "Inclusion",
    "Benefit", "Health", "Insurance", "Remote", "Hybrid", "Onsite", "Office", "City", "State",
    "United", "States", "Global", "Global", "Department", "Manager", "Director", "Senior",
    "Junior", "Lead", "Head", "Level", "Full", "Part", "Time",
}


def known_aliases_lower():
    known = set()
    for canonical_map in TECH_DICTIONARY.values():
        for canonical, aliases in canonical_map.items():
            known.add(canonical.lower())
            for alias in aliases:
                known.add(alias.lower())
    return known


def mine(df, text_col="description", company_col="company", min_count=5, max_lowercase_ratio=0.3):
    known = known_aliases_lower()
    # 회사 소개/자사 언급 노이즈 제거: 이 데이터셋에 등장하는 채용 회사명 자체는 기술스택이 아니다.
    known_companies = set()
    if company_col and company_col in df.columns:
        known_companies = {c.lower() for c in df[company_col].dropna().unique()}

    counter = Counter()
    lowercase_counter = Counter()
    for text in df[text_col].dropna():
        for m in CANDIDATE_PATTERN.finditer(text):
            token = m.group()
            token_lower = token.lower()
            if token in STOPWORDS or token_lower in known or token_lower in known_companies:
                continue
            if len(token) < 2:
                continue
            counter[token] += 1
        for m in LOWERCASE_WORD_PATTERN.finditer(text):
            lowercase_counter[m.group()] += 1

    rows = []
    for tok, cnt in counter.items():
        if cnt < min_count:
            continue
        lower_cnt = lowercase_counter.get(tok.lower(), 0)
        if lower_cnt > max_lowercase_ratio * cnt:
            continue  # 소문자로도 흔히 쓰이는 일반 단어로 판단, 제외
        rows.append((tok, cnt, lower_cnt))

    rows.sort(key=lambda x: -x[1])
    return pd.DataFrame(rows, columns=["candidate", "count", "lowercase_count"])


### 후보 마이닝 실행

`input_csv`를 채용공고 데이터 CSV 경로로 바꿔서 실행하세요.


In [4]:
input_csv = "dev_role_jobs.csv"
candidates_output_csv = "tech_candidates.csv"
text_col = "description"
min_count = 5

df = pd.read_csv(input_csv)
candidates = mine(df, text_col=text_col, min_count=min_count)
#candidates.to_csv(candidates_output_csv, index=False)

#print(f"후보 {len(candidates)}개를 {candidates_output_csv} 에 저장했습니다.")
print("csv파일 열어서 진짜 기술스택만 골라 1단계 TECH_DICTIONARY 에 추가하세요.")
print()
print(candidates.head(50).to_string(index=False))


csv파일 열어서 진짜 기술스택만 골라 1단계 TECH_DICTIONARY 에 추가하세요.

       candidate  count  lowercase_count
              AI  15155               41
             San   1687                0
        Internet   1623              301
       Francisco   1514                0
              ML   1432                2
            Fair   1273              149
          Chance   1271               22
            APIs    942                0
     Familiarity    863              222
       Ordinance    862                0
             Los    841                0
         Angeles    841                0
          County    819                0
             LLM    749                3
             API    747                3
             USD    708                0
             Act    661              194
       Applicant    634               50
              CI    538                0
         Fortune    529                1
         However    459               93
           Atlas    458                0
     

## 3. 기술 태그 추출 (`extract_tech_tags.py`)

description 열에서 1단계 사전을 기반으로 기술 스택을 뽑아
tech_stack 열(콤마로 구분된 canonical name 문자열)로 저장한다.


In [5]:
# 일반 토큰의 경계: 문자/숫자/밑줄이 아니면 경계로 취급 (대소문자 무시 매칭)
_WORD = r"A-Za-z0-9_"
# ambiguous 토큰(Go, R, C, D)의 경계는 더 엄격하게 잡는다.
# "R&D"의 R, "A/B"의 B, "C-suite"/"C‑suite"(특수 대시)의 C, "Series A–C"의 C(en dash),
# "D.C."의 D, "WE'D"/"WE'D"(아포스트로피 축약형)의 D 처럼 약어·합성어·축약형 안에 낀
# 한 글자를 오매칭하지 않도록 &, /, ., 아포스트로피, 각종 대시도 "단어의 일부"로 취급해서 제외한다.
_WORD_EXT = r"A-Za-z0-9_&/.'’‐‑‒–—―\-"

# 그래도 남는 "Go to Market", "Series C" 같은 오탐은 문자 경계만으로 못 거른다.
# 실제 기술스택 나열은 거의 항상 다른(비-ambiguous) 기술명과 가까이 등장하므로,
# ambiguous 매칭 주변 이 범위 안에 정상 매칭이 하나도 없으면 버린다.
_AMBIGUOUS_CONTEXT_WINDOW = 50


def _flatten_dictionary():
    normal_lookup = {}  # alias.lower() -> canonical
    ambiguous_lookup = {}  # alias(대소문자 그대로) -> canonical
    normal_aliases = []
    ambiguous_aliases = []
    for canonical_map in TECH_DICTIONARY.values():
        for canonical, aliases in canonical_map.items():
            for alias in aliases:
                if alias in AMBIGUOUS_EXACT:
                    ambiguous_lookup[alias] = canonical
                    ambiguous_aliases.append(alias)
                else:
                    normal_lookup[alias.lower()] = canonical
                    normal_aliases.append(alias)
    return normal_lookup, ambiguous_lookup, normal_aliases, ambiguous_aliases


def _build_pattern(aliases, boundary_class, flags=0):
    # 같은 위치에서 더 구체적인(긴) 표현을 먼저 시도하도록 길이 내림차순 정렬
    # 예: "JavaScript"를 "Java"보다 먼저 시도해야 "JavaScript" 안의 "Java"를 잘못 떼어내지 않음
    aliases_sorted = sorted(set(aliases), key=len, reverse=True)
    escaped = [re.escape(a) for a in aliases_sorted]
    left = rf"(?<![{boundary_class}])"
    right = rf"(?![{boundary_class}])"
    return re.compile(left + "(?:" + "|".join(escaped) + ")" + right, flags)


_NORMAL_LOOKUP, _AMBIGUOUS_LOOKUP, _NORMAL_ALIASES, _AMBIGUOUS_ALIASES = _flatten_dictionary()
_NORMAL_PATTERN = _build_pattern(_NORMAL_ALIASES, _WORD, flags=re.IGNORECASE)
_AMBIGUOUS_PATTERN = _build_pattern(_AMBIGUOUS_ALIASES, _WORD_EXT)


def extract_tags(text):
    """텍스트 하나에서 canonical 기술명 리스트(정렬, 중복 제거)를 반환."""
    if not isinstance(text, str) or not text:
        return []
    found = set()

    normal_spans = list(_NORMAL_PATTERN.finditer(text))
    for m in normal_spans:
        found.add(_NORMAL_LOOKUP[m.group().lower()])

    for m in _AMBIGUOUS_PATTERN.finditer(text):
        s, e = m.span()
        window_start, window_end = s - _AMBIGUOUS_CONTEXT_WINDOW, e + _AMBIGUOUS_CONTEXT_WINDOW
        has_nearby_tech = any(
            n.start() < window_end and n.end() > window_start for n in normal_spans
        )
        if has_nearby_tech:
            found.add(_AMBIGUOUS_LOOKUP[m.group()])

    return sorted(found)


def tag_dataframe(df, text_col="description", tag_col="tech_stack", company_col="company"):
    """
    company_col 이 주어지면, 채용 회사 이름과 canonical 기술명이 같은 태그는 제거한다.
    (예: Cloudflare/Datadog/OpenAI 자사 채용공고의 "Who we are" 소개 문단에서
    회사 이름 자체가 기술스택으로 잘못 잡히는 것을 방지)
    """
    df = df.copy()

    def _row_tags(text, company):
        tags = extract_tags(text)
        if company_col and isinstance(company, str) and company.strip():
            company_lower = company.strip().lower()
            # "MongoDB" 회사가 자사 제품 "MongoDB Atlas"를 소개하는 것처럼, 태그가
            # 회사명 전체/앞단어를 포함하거나 그 반대인 경우까지 자기 언급으로 보고 제거한다.
            company_first_word = company_lower.split()[0] if company_lower.split() else company_lower
            tags = [
                t for t in tags
                if company_lower not in t.lower()
                and t.lower() not in company_lower
                and company_first_word not in t.lower()
            ]
        return ", ".join(tags)

    if company_col and company_col in df.columns:
        df[tag_col] = [
            _row_tags(t, c) for t, c in zip(df[text_col], df[company_col])
        ]
    else:
        df[tag_col] = df[text_col].apply(lambda t: _row_tags(t, None))
    return df


### 태그 추출 실행

2단계에서 사전을 다 채운 뒤, 원본 CSV에 `tech_stack` 열을 추가해 저장합니다.


In [7]:
tag_input_csv = "dev_role_jobs.csv"
tag_output_csv = "dev_role_jobs_tagged.csv"
tag_text_col = "description"

df = pd.read_csv(tag_input_csv)
tagged = tag_dataframe(df, text_col=tag_text_col)
tagged.to_csv(tag_output_csv, index=False)

non_empty = (tagged["tech_stack"] != "").sum()
print(f"{len(tagged):,}건 중 {non_empty:,}건({non_empty / len(tagged):.1%})에서 태그 추출됨")
print()
print("태그 빈도 Top 30")
print(tagged["tech_stack"].str.split(", ").explode().value_counts().head(30))


2,613건 중 2,112건(80.8%)에서 태그 추출됨

태그 빈도 Top 30
tech_stack
Python          1183
AWS              563
                 501
Kubernetes       498
Go               482
GCP              458
TypeScript       455
Java             430
Azure            426
C++              396
JavaScript       389
SQL              360
GPT              355
C                326
Rust             273
React            272
Claude           197
Linux            179
Terraform        178
PostgreSQL       167
Node.js          154
Docker           152
GitHub           150
Apache Spark     142
Apache Kafka     124
PyTorch          106
C#                83
Ruby              81
MCP               73
Airflow           70
Name: count, dtype: int64


### tech stack 비어있는 케이스 분석

- 여기서 케이스 분석 후 루프

In [8]:
dev_role = pd.read_csv("/mnt/c/Users/hojin/OneDrive/바탕 화면/ybig_team/experiment/dev_role_jobs_tagged.csv")
dev_role.head()

,company,title,description,normalized_title,job_role,tech_stack
0,Cohere,"Member of Technical Staff, MLE (Korea)",Who are we?\n\nCohere is the leading security-...,"member of technical staff, mle",General Software Engineer,"JAX, Python, TensorFlow"
1,Cohere,"Member of Technical Staff, Modeling",Who are we?\n\nCohere is the leading security-...,"member of technical staff, modeling",General Software Engineer,"CUDA, JAX, Python, TensorFlow, Transformers"
2,Cohere,"Senior Member of Technical Staff, Multimodal AI",Who are we?\n\nCohere is the leading security-...,"senior member of technical staff, multimodal ai",ML / AI Engineer / Data Scientist,"CUDA, Hugging Face, JAX, Llama, PyTorch, Pytho..."
3,Cohere,Data Annotation Specialist - German Writer/Tra...,Who are we?\n\nOur mission is to scale intelli...,data annotation specialist german writer trans...,ML / AI Engineer / Data Scientist,NaN
4,Cohere,"Software Engineer, Security",Who are we?\n\nCohere is the leading security-...,"software engineer, security",General Software Engineer,"AWS, Azure, GCP, Kubernetes, Python"


In [9]:
dev_role['tech_stack'].isna().sum()

np.int64(501)

In [10]:
## tech stack이 비어있는 데이터만 검출

dev_role_empty = dev_role[dev_role['tech_stack'].isna()]
dev_role_empty

,company,title,description,normalized_title,job_role,tech_stack
3,Cohere,Data Annotation Specialist - German Writer/Tra...,Who are we?\n\nOur mission is to scale intelli...,data annotation specialist german writer trans...,ML / AI Engineer / Data Scientist,NaN
17,Cohere,"Research Scientist, Cohere Labs",Who are we?\n\nCohere is the leading security-...,"research scientist, cohere labs",Research Engineer / Scientist,NaN
19,Cohere,"Senior Research Scientist, Model Evaluation",Who are we?\n\nCohere is the leading security-...,"senior research scientist, model evaluation",Research Engineer / Scientist,NaN
23,Cohere,Senior Security Engineer,Who are we?\n\nCohere is the leading security-...,senior security engineer,Security Engineer,NaN
25,Cohere,"Staff Research Engineer, Model Efficiency",Who are we?\n\nCohere is the leading security-...,"staff research engineer, model efficiency",Research Engineer / Scientist,NaN
...,...,...,...,...,...,...
2589,Spotify,"Machine Learning Engineering Manager, Personal...","Lead, coach, and develop a team of machine lea...","machine learning engineering manager, personal...",Engineering Management,NaN
2593,Spotify,Research Scientist - Personalization,The Personalization team makes deciding what t...,research scientist personalization,Research Engineer / Scientist,NaN
2599,Spotify,"Senior Machine Learning Engineer, Personalizat...",The Personalization team makes deciding what t...,"senior machine learning engineer, personalizat...",General Software Engineer,NaN
2610,Spotify,Staff Machine Learning Engineer - Policy & Safety,We design Spotify’s consumer experience—end to...,staff machine learning engineer policy & safety,ML / AI Engineer / Data Scientist,NaN


In [11]:
dev_role_empty.to_csv('dev_role_empty.csv', index=False)